# Reproduce *Do VLMs Share Safety Neurons Across Modalities?*

This Colab notebook runs the authors' released safety-neuron detection pipeline on **Qwen3-VL-2B**, the smallest model in the paper. It runs the text and visual pipelines separately, then compares the confirmed neurons.

The notebook pins the inspected repository code at commit `b983d5641861a87333bb04e4359470ed2d35649a`. The research commands and hyperparameters come from the repository; the surrounding cells only prepare Colab, save results, and display outputs.

> **Scope:** the public repository contains the neuron-detection pipeline. It does not contain scripts for the paper's complete ViSafe-Eval intervention tables or GEVD subspace experiments, so this notebook does not claim to reproduce those results.

> **Content warning:** the datasets contain harmful text and images.

## 0. Before running

1. In Colab, select **Runtime → Change runtime type → GPU**. Use a high-memory runtime if available. The repository says the authors tested the 2B–9B models on one 80 GB GPU; Qwen3-VL-2B visual detection also keeps about 9 GB of activations in CPU memory. Colab hardware is not guaranteed, so the full run may not fit every runtime.
2. Create a Hugging Face token and add it to **Colab → Secrets** as `HF_TOKEN`.
3. Before fetching the data, accept the access conditions for the gated datasets listed in the repository README: [HarmBench](https://huggingface.co/datasets/walledai/HarmBench), [AdvBench](https://huggingface.co/datasets/walledai/AdvBench), [UnsafeBench](https://huggingface.co/datasets/yiting/UnsafeBench), [VLGuard](https://huggingface.co/datasets/ys-zong/VLGuard), and [HoliSafe](https://huggingface.co/datasets/etri-vilab/holisafe-bench).
4. For the paper's exact labeling procedure, add `AZURE_OPENAI_ENDPOINT` and `AZURE_OPENAI_API_KEY` to Colab Secrets. The authors' code uses an Azure deployment named `gpt-4o`. Without these credentials, use keyword-only labeling; that verifies the pipeline but is not an exact reproduction of the paper's labels.

In [ ]:
# Check the Colab hardware before downloading anything.
import os, platform, subprocess, sys

print('Notebook Python:', sys.version)
print('Platform:', platform.platform())
subprocess.run(['nvidia-smi'], check=False)

with open('/proc/meminfo') as f:
    mem_kb = int(next(line for line in f if line.startswith('MemTotal:')).split()[1])
print(f'CPU RAM: {mem_kb / 1024**2:.1f} GiB')

## 1. Download the inspected repository version

This uses your GitHub fork and checks out the exact code version inspected for this notebook.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/tropaafsara/VLM-safety-neuron.git'
REPO_COMMIT = 'b983d5641861a87333bb04e4359470ed2d35649a'
REPO_DIR = Path('/content/VLM-safety-neuron')

def run(command, cwd=None, env=None):
    print('$', ' '.join(map(str, command)))
    subprocess.run(list(map(str, command)), cwd=cwd, env=env, check=True)

if not REPO_DIR.exists():
    run(['git', 'clone', REPO_URL, REPO_DIR])
run(['git', 'fetch', 'origin'], cwd=REPO_DIR)
run(['git', 'checkout', REPO_COMMIT], cwd=REPO_DIR)
run(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR)

## 2. Create the authors' tested Python environment

The README says the code was tested with Python 3.10. For Qwen3-VL-2B it specifies `torch==2.4.0`, `torchvision==0.19.0`, and permits tagged `transformers==5.7.0`. This cell creates an isolated Python 3.10 environment and installs those versions plus the repository requirements.

In [ ]:
# uv is used only to create an isolated Python 3.10 environment in Colab.
run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])

ENV_DIR = Path('/content/vlm-safety-env')
if not ENV_DIR.exists():
    run(['uv', 'venv', '--python', '3.10', ENV_DIR])
PYTHON = str(ENV_DIR / 'bin' / 'python')

run(['uv', 'pip', 'install', '--python', PYTHON,
     'torch==2.4.0', 'torchvision==0.19.0', 'transformers==5.7.0'])
run(['uv', 'pip', 'install', '--python', PYTHON,
     '-r', REPO_DIR / 'requirements.txt'])

In [ ]:
# Verify the versions and GPU access from the isolated environment.
verify = (
    "import sys, torch, torchvision, transformers; "
    "print('Python', sys.version); "
    "print('torch', torch.__version__); "
    "print('torchvision', torchvision.__version__); "
    "print('transformers', transformers.__version__); "
    "print('CUDA available:', torch.cuda.is_available()); "
    "print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'); "
    "print('GPU memory GiB:', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1) if torch.cuda.is_available() else 0)"
)
run([PYTHON, '-c', verify])

## 3. Save long-run results in Google Drive

The detection runs can be long. Saving outputs in Drive prevents losing completed rounds when the Colab session disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RESULTS_ROOT = Path('/content/drive/MyDrive/VLM-safety-neuron-results')
TEXT_OUT = RESULTS_ROOT / 'text' / 'qwen3vl-2b'
VISUAL_OUT = RESULTS_ROOT / 'visual' / 'qwen3vl-2b'
FIGURE_DIR = RESULTS_ROOT / 'figures'
for path in (TEXT_OUT, VISUAL_OUT, FIGURE_DIR):
    path.mkdir(parents=True, exist_ok=True)
print('Results will be saved under:', RESULTS_ROOT)

## 4. Load the Hugging Face token

In [ ]:
from getpass import getpass
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None
if not hf_token:
    hf_token = getpass('Paste your Hugging Face token: ')
if not hf_token:
    raise RuntimeError('HF_TOKEN is required to access the model and gated datasets.')

os.environ['HF_TOKEN'] = hf_token
os.environ['HF_HOME'] = '/content/hf_cache'
print('HF_TOKEN loaded without displaying it.')

## 5. Choose the behavioral-labeling mode

- `USE_AZURE_JUDGE = True` follows the paper's released pipeline: the 17-template keyword pass is followed by GPT-4o re-judging keyword-COMPLY responses.
- `USE_AZURE_JUDGE = False` adds the repository's `--no-judge` flag and uses keyword labels only. Use this to test the pipeline when Azure GPT-4o is unavailable, but do not call its neuron counts an exact paper reproduction.

In [ ]:
USE_AZURE_JUDGE = False  # Change to True only if both Azure secrets are configured.

if USE_AZURE_JUDGE:
    try:
        os.environ['AZURE_OPENAI_ENDPOINT'] = userdata.get('AZURE_OPENAI_ENDPOINT')
        os.environ['AZURE_OPENAI_API_KEY'] = userdata.get('AZURE_OPENAI_API_KEY')
    except Exception as exc:
        raise RuntimeError('Add both Azure OpenAI values to Colab Secrets.') from exc
    if not os.environ.get('AZURE_OPENAI_ENDPOINT') or not os.environ.get('AZURE_OPENAI_API_KEY'):
        raise RuntimeError('Both Azure OpenAI secrets are required when USE_AZURE_JUDGE=True.')
    JUDGE_ARGS = []
    print('Mode: keyword pass + Azure GPT-4o judge (paper pipeline)')
else:
    JUDGE_ARGS = ['--no-judge']
    print('Mode: keyword-only labels (pipeline test, not exact paper labels)')

## 6. Build ViSafe-Detect with the authors' script

The text pipeline downloads its 900 harmful and 900 safe rows automatically. The visual pipeline requires this separate fetch step for 2,417 harmful and 2,417 safe images. If this cell reports an access error, open the relevant Hugging Face dataset page, accept its conditions, and run the cell again.

In [ ]:
run([PYTHON, 'data/build/fetch_visafe_detect.py'], cwd=REPO_DIR, env=os.environ.copy())

In [ ]:
# Confirm that the repository scripts and rebuilt visual labels are present.
required = [
    REPO_DIR / 'src/run_iterative_text.py',
    REPO_DIR / 'src/run_iterative_visual.py',
    REPO_DIR / 'data/text/text_detection.jsonl',
    REPO_DIR / 'data/visafe_detect/labels.jsonl',
]
for path in required:
    print(('OK      ' if path.exists() else 'MISSING '), path)
assert all(path.exists() for path in required), 'A required paper file is missing.'

def line_count(path):
    with open(path, 'rb') as f:
        return sum(1 for _ in f)

print('Text manifest rows:', line_count(REPO_DIR / 'data/text/text_detection.jsonl'))
print('ViSafe-Detect label rows:', line_count(REPO_DIR / 'data/visafe_detect/labels.jsonl'))

## 7. Optional visual smoke test

This uses only arguments already supported by the authors' visual script: one round, 100 harmful images, 100 safe images, and keyword-only labels. It checks that model loading, activation extraction, probing, and patching can run on your Colab hardware. Its result is **not** a reproduction result. Skip this cell if you want to go directly to the full run.

In [ ]:
RUN_SMOKE_TEST = False  # Change to True to run this optional test.

if RUN_SMOKE_TEST:
    smoke_out = RESULTS_ROOT / 'smoke' / 'visual-qwen3vl-2b'
    cmd = [
        PYTHON, 'src/run_iterative_visual.py',
        '--model', 'qwen3vl-2b',
        '--output-dir', smoke_out,
        '--n-rounds', '1',
        '--n-harmful', '100',
        '--n-safe', '100',
        '--seed', '42',
        '--no-judge',
    ]
    run(cmd, cwd=REPO_DIR, env=os.environ.copy())
else:
    print('Smoke test skipped.')

## 8. Full text safety-neuron detection

This is the released text pipeline for Qwen3-VL-2B: 900 harmful queries, 900 safe queries, a stratified 80/20 split, three iterative rounds, seed 42, probing, mean-activation patching, and permanent clamping of confirmed neurons between rounds.

Change `RUN_FULL_TEXT` to `True`, then run the cell.

In [ ]:
RUN_FULL_TEXT = False

if RUN_FULL_TEXT:
    text_cmd = [
        PYTHON, 'src/run_iterative_text.py',
        '--model', 'qwen3vl-2b',
        '--output-dir', TEXT_OUT,
        '--n-rounds', '3',
        '--seed', '42',
        *JUDGE_ARGS,
    ]
    run(text_cmd, cwd=REPO_DIR, env=os.environ.copy())
else:
    print('Set RUN_FULL_TEXT=True when you are ready.')

## 9. Full visual safety-neuron detection

This is the released visual pipeline for Qwen3-VL-2B using all ViSafe-Detect images and the same three-round iterative procedure. It stores the image activations for each round, so this is the most memory-intensive part.

Change `RUN_FULL_VISUAL` to `True`, then run the cell.

In [ ]:
RUN_FULL_VISUAL = False

if RUN_FULL_VISUAL:
    visual_cmd = [
        PYTHON, 'src/run_iterative_visual.py',
        '--model', 'qwen3vl-2b',
        '--output-dir', VISUAL_OUT,
        '--n-rounds', '3',
        '--seed', '42',
        *JUDGE_ARGS,
    ]
    run(visual_cmd, cwd=REPO_DIR, env=os.environ.copy())
else:
    print('Set RUN_FULL_VISUAL=True when you are ready.')

## 10. Resume an interrupted round

The authors' scripts save responses before judging. If a run stops during round 2 or 3, set the correct round below. The script checks that the earlier configuration and outputs match before resuming. Leave the value at `0` when no resume is needed.

In [ ]:
RESUME_TEXT_ROUND = 0   # Use 2 or 3 only when that round was interrupted.
RESUME_VISUAL_ROUND = 0 # Use 2 or 3 only when that round was interrupted.

if RESUME_TEXT_ROUND:
    run([PYTHON, 'src/run_iterative_text.py', '--model', 'qwen3vl-2b',
         '--output-dir', TEXT_OUT, '--n-rounds', '3', '--seed', '42',
         '--start-round', str(RESUME_TEXT_ROUND), *JUDGE_ARGS],
        cwd=REPO_DIR, env=os.environ.copy())

if RESUME_VISUAL_ROUND:
    run([PYTHON, 'src/run_iterative_visual.py', '--model', 'qwen3vl-2b',
         '--output-dir', VISUAL_OUT, '--n-rounds', '3', '--seed', '42',
         '--start-round', str(RESUME_VISUAL_ROUND), *JUDGE_ARGS],
        cwd=REPO_DIR, env=os.environ.copy())

if not RESUME_TEXT_ROUND and not RESUME_VISUAL_ROUND:
    print('No resume requested.')

## 11. Compare the reproduced neuron counts

The paper reports for Qwen3-VL-2B:

- **Text (Table 3):** rounds `34 + 10 + 4 = 48` confirmed neurons.
- **Visual (Table 4):** rounds `3 + 0 + 1 = 4` confirmed neurons.

Use these as the paper values for comparison. If you used `--no-judge`, differences do not constitute a failed exact reproduction because the behavioral labels are different from the paper's judge-assisted labels.

In [ ]:
import json

def show_run(name, output_dir):
    summary_path = output_dir / 'iterative_summary.json'
    neurons_path = output_dir / 'all_confirmed_neurons.json'
    print(f'\n{name}: {output_dir}')
    if not summary_path.exists() or not neurons_path.exists():
        print('  Result files are not available yet.')
        return None
    summary = json.loads(summary_path.read_text())
    neurons_data = json.loads(neurons_path.read_text())
    neurons = neurons_data.get('neurons', neurons_data) if isinstance(neurons_data, dict) else neurons_data
    for rnd in summary.get('rounds', []):
        print(f"  round {rnd['round']}: status={rnd['status']}, "
              f"confirmed={rnd.get('stage2_confirmed', 0)}, "
              f"total={rnd.get('total_confirmed', 0)}, "
              f"labels={rnd.get('label_method', 'unknown')}")
    print('  total confirmed:', len(neurons))
    return len(neurons)

text_count = show_run('TEXT', TEXT_OUT)
visual_count = show_run('VISUAL', VISUAL_OUT)
print('\nPaper comparison for Qwen3-VL-2B: text=48, visual=4')
if text_count is not None and visual_count is not None:
    print(f'Your run: text={text_count}, visual={visual_count}')

## 12. Plot the neuron locations with the authors' plotting script

In [ ]:
from IPython.display import Image, display

plot_jobs = [
    ('text', TEXT_OUT, FIGURE_DIR / 'qwen3vl-2b_text.png'),
    ('visual', VISUAL_OUT, FIGURE_DIR / 'qwen3vl-2b_visual.png'),
]
for modality, output_dir, figure_path in plot_jobs:
    neuron_path = output_dir / 'all_confirmed_neurons.json'
    if not neuron_path.exists():
        print(f'Skipping {modality}: {neuron_path} is not available.')
        continue
    run([PYTHON, 'src/plot_neuron_distribution.py',
         '--neurons', neuron_path, '--model', 'qwen3vl-2b',
         '--title', f'Qwen3-VL-2B ({modality} safety)',
         '--output', figure_path], cwd=REPO_DIR, env=os.environ.copy())
    display(Image(filename=str(figure_path)))

## 13. Export the small result files

This creates a compact ZIP containing summaries, confirmed-neuron lists, and figures. It intentionally leaves out the large saved activation arrays.

In [ ]:
import shutil
from google.colab import files

bundle_dir = Path('/content/qwen3vl-2b-reproduction-summary')
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
bundle_dir.mkdir(parents=True)

for modality, output_dir in [('text', TEXT_OUT), ('visual', VISUAL_OUT)]:
    target = bundle_dir / modality
    target.mkdir()
    for filename in ('config.json', 'iterative_summary.json', 'all_confirmed_neurons.json'):
        source = output_dir / filename
        if source.exists():
            shutil.copy2(source, target / filename)
    for round_dir in sorted(output_dir.glob('round_*')):
        source = round_dir / 'round_summary.json'
        if source.exists():
            shutil.copy2(source, target / f'{round_dir.name}_summary.json')

if FIGURE_DIR.exists():
    shutil.copytree(FIGURE_DIR, bundle_dir / 'figures', dirs_exist_ok=True)

zip_path = shutil.make_archive('/content/qwen3vl-2b-reproduction-summary', 'zip', bundle_dir)
print('Created:', zip_path)
files.download(zip_path)

## What counts as a successful reproduction?

For this released-code reproduction, report:

1. the repository commit, environment versions, model, seed, and labeling mode;
2. the confirmed text neurons per round and total;
3. the confirmed visual neurons per round and total;
4. their layer/component plots; and
5. whether the qualitative asymmetry holds: many more localized text-safety neurons than visual-safety neurons.

Do not report the optional 100-image smoke test as a paper reproduction. Do not claim that this notebook reproduces the ViSafe-Eval ablation effects or GEVD analysis, because those experiment scripts are absent from the released repository.